# Notebook 09: Interactive Dashboard Guide
## Forecasting Education Performance -- Tanzania BEST Datasets (2020-2024)
**Author:** Habil Masawika | **Project:** Tanzania BEST ML Forecasting

---

### Overview
This notebook documents the Streamlit dashboard (`app/streamlit_app.py`) and generates
static preview screenshots of the dashboard panels. The interactive dashboard allows
non-technical users to explore BEST data, generate predictions, compare regions, and
download reports.

### Dashboard Features
- Upload BEST datasets for any year
- Select regions and years for comparison
- Generate ML predictions for selected inputs
- Visualise trends interactively
- Download prediction reports as CSV
- Inspect feature importance charts

In [ ]:
import sys, warnings
sys.path.insert(0, '/home/claude/BEST-ML-Forecasting/src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from utilities import setup_logging, set_seeds, ProjectPaths
from visualization import (plot_csee_trend, plot_enrolment_by_region,
                            plot_ptr_by_region, plot_electricity_by_region)

set_seeds(42)
paths = ProjectPaths()

panel_fe  = pd.read_csv(paths.processed('best_panel_features.csv'))
csee_nat  = pd.read_csv(paths.processed('csee_national_trend.csv'))
print("Dashboard data loaded.")

In [ ]:
# ── Static dashboard preview: Overview panel ─────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Panel 1: CSEE trend
df_csee = csee_nat[csee_nat['year'] >= 2015].sort_values('year')
axes[0,0].fill_between(df_csee['year'], df_csee['csee_pass_rate'], alpha=0.2, color='steelblue')
axes[0,0].plot(df_csee['year'], df_csee['csee_pass_rate'], 'o-', color='steelblue', lw=2.5, ms=8)
for _, r in df_csee.iterrows():
    axes[0,0].annotate(f"{r.csee_pass_rate:.1f}%", (r.year, r.csee_pass_rate),
                        textcoords='offset points', xytext=(0,8), ha='center', fontsize=8)
axes[0,0].set_title('CSEE National Pass Rate Trend', fontweight='bold')
axes[0,0].set_ylabel('Pass Rate (%)')
axes[0,0].grid(axis='y', alpha=0.3)

# Panel 2: Enrolment by region (2023)
df_e = panel_fe[panel_fe['year']==2023].dropna(subset=['enrolment_f1f4'])
df_e = df_e.nlargest(10, 'enrolment_f1f4').sort_values('enrolment_f1f4')
axes[0,1].barh(df_e['region'], df_e['enrolment_f1f4']/1e3, color='darkorange', alpha=0.85)
axes[0,1].set_title('Top 10 Regions by Enrolment (2023)', fontweight='bold')
axes[0,1].set_xlabel("Enrolment (000s)")

# Panel 3: PTR distribution
ptr_data = panel_fe.dropna(subset=['ptr_regional'])
years = sorted(ptr_data['year'].unique())
data = [ptr_data[ptr_data['year']==yr]['ptr_regional'].values for yr in years]
bp = axes[1,0].boxplot(data, labels=years, patch_artist=True,
                        medianprops=dict(color='black', lw=2))
for patch in bp['boxes']: patch.set_facecolor('steelblue'); patch.set_alpha(0.7)
axes[1,0].axhline(30, color='red', lw=1.5, linestyle='--', label='30:1 threshold')
axes[1,0].set_title('Regional PTR Distribution by Year', fontweight='bold')
axes[1,0].set_ylabel('Pupil-Teacher Ratio')
axes[1,0].legend()

# Panel 4: Electricity by region
reg_e = panel_fe.groupby('region')['pct_schools_electricity'].mean().nsmallest(10)
colours_e = ['darkorange' if v < 70 else 'mediumseagreen' for v in reg_e.values]
axes[1,1].barh(reg_e.index, reg_e.values, color=colours_e, alpha=0.85)
axes[1,1].axvline(70, color='green', lw=1.5, linestyle='--')
axes[1,1].set_title('10 Lowest Electricity Access Regions', fontweight='bold')
axes[1,1].set_xlabel('% Schools with Electricity')

plt.suptitle('Tanzania BEST Education Dashboard -- Overview Panel',
             fontweight='bold', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(paths.fig('nb09_dashboard_preview.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Dashboard preview saved.")

In [ ]:
# ── Prediction widget demo ────────────────────────────────────────────────
from utilities import load_model

scaler     = load_model(paths.model('feature_scaler.pkl'))
best_model = load_model(paths.model('gradient_boosting.pkl'))
model_feats = list(pd.read_csv(paths.table('model_features.csv'))['feature'])

# Demo: predict for a hypothetical input
demo_input = {f: panel_fe[f].mean() for f in model_feats if f in panel_fe.columns}
x_demo = pd.DataFrame([demo_input])[model_feats].fillna(panel_fe[model_feats].mean())
x_scaled = scaler.transform(x_demo.values)
pred = best_model.predict(x_scaled)[0]
print(f"Demo prediction (mean feature values): {pred:.2f}%")
print("This is the predicted CSEE pass rate for a region with mean system inputs.")

In [ ]:
# ── Dashboard launch instructions ─────────────────────────────────────────
print('STREAMLIT DASHBOARD')
print('===================')
print('The interactive dashboard is located at: app/streamlit_app.py')
print('')
print('To launch locally:')
print('  1. Install Streamlit:  pip install streamlit')
print('  2. Run:  streamlit run app/streamlit_app.py')
print('')
print('Dashboard panels:')
print('  1. Overview      -- National trends in key indicators')
print('  2. Regional Deep Dive  -- Select a region to see its full profile')
print('  3. Predictions   -- Enter feature values to generate model predictions')
print('  4. Forecast      -- View 2025-2030 scenario forecasts')
print('  5. Data Explorer -- Filter, sort, and download the cleaned dataset')
print('  6. Model Info    -- Feature importance and model performance metrics')
print('')
print('The app also supports CSV upload for users who have new BEST data files.')